# LeetCode 364 — Nested List Weight Sum II bằng Python

Link đề: https://leetcode.com/problems/nested-list-weight-sum-ii/ · Độ khó: **Medium** · Chủ đề: Stack, Depth-First Search, Breadth-First Search

Skeleton LeetCode:

```python
# """
# This is the interface that allows for creating nested lists.
# You should not implement it, or speculate about its implementation
# """
# class NestedInteger:
#    def __init__(self, value=None):
#    def isInteger(self):
#    def add(self, elem):
#    def setInteger(self, value):
#    def getInteger(self):
#    def getList(self):
class Solution:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        ...
```

Ghi chú: đây là bài **Premium** nên đề được viết theo hiểu biết về bài; hãy đối chiếu lại với đề gốc nếu bạn có tài khoản Premium. LeetCode cung cấp sẵn lớp `NestedInteger`; trong notebook này ta **tự dựng** một bản tương đương (chỉ để chạy thử) và hàm `to_nested` biến list Python thành `NestedInteger`. Hai ví dụ trong đề (`[[1,1],2,[1,1]] → 8` và `[1,[4,[6]]] → 17`) khớp bản doocs; nếu bạn thấy khác đề gốc, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi số nguyên có **độ sâu** (số lớp danh sách bao quanh nó) và **trọng số đảo ngược** `maxDepth − depth + 1` — số **càng nông** thì **trọng số càng lớn** (ngược với bài 339).
2. Nắm các công cụ Python: **đệ quy** trên cấu trúc lồng nhau, từ khóa `nonlocal`, `collections.deque` cho BFS, `list.extend`, và cách dùng giao diện `NestedInteger` (`isInteger`, `getInteger`, `getList`).
3. Đi từ cách **hai lượt** (tìm `maxDepth` rồi mới tính tổng) → **một lượt bằng công thức** `(maxDepth + 1)·S − WS` → **BFS cộng dồn theo từng tầng** → **gom tổng theo độ sâu rồi nhân trọng số**.
4. Hiểu mấu chốt: trọng số `maxDepth − depth + 1` **chỉ biết được sau khi đã duyệt xong** (cần `maxDepth`), nên ta hoặc **duyệt hai lần**, hoặc **tách tổng ra** thành `(maxDepth + 1)·Σa − Σa·depth`, hoặc dùng mẹo **cộng dồn theo tầng** để không cần biết `maxDepth` trước.
5. Tránh các lỗi kinh điển: dùng `depth` làm trọng số (nhầm sang bài 339), lệch 1 khi đếm độ sâu, quên `nonlocal`, gọi `getInteger()` trên một danh sách (trả `None`).

## 2. Đề bài & ví dụ

Cho một **danh sách lồng nhau** `nestedList` các số nguyên. Mỗi phần tử là **một số nguyên** hoặc **một danh sách** mà các phần tử của nó cũng có thể là số nguyên hoặc danh sách khác.

**Độ sâu (depth)** của một số nguyên là **số danh sách chứa nó**. Ví dụ `[1,[2,2],[[3],2],1]`: số `1` ở ngoài cùng có độ sâu `1`, hai số `2` trong `[2,2]` có độ sâu `2`, số `3` trong `[[3],2]` có độ sâu `3`...

Gọi `maxDepth` là **độ sâu lớn nhất** của một số nguyên nào đó. **Trọng số (weight)** của một số nguyên là `maxDepth − (độ sâu của nó) + 1`.

Trả về **tổng** của mọi số nguyên nhân với **trọng số** của nó.

**Ví dụ 1**

- Input: `nestedList = [[1,1],2,[1,1]]`
- Output: `8`
- Giải thích: bốn số `1` ở độ sâu `2` (= `maxDepth`) nên trọng số `1`; số `2` ở độ sâu `1` nên trọng số `2`. Tổng `1·1 + 1·1 + 2·2 + 1·1 + 1·1 = 8`.

**Ví dụ 2**

- Input: `nestedList = [1,[4,[6]]]`
- Output: `17`
- Giải thích: `1` ở độ sâu `1`, `4` ở độ sâu `2`, `6` ở độ sâu `3` (= `maxDepth`). Trọng số lần lượt `3, 2, 1`. Tổng `1·3 + 4·2 + 6·1 = 17`.

**Ràng buộc:** `1 <= nestedList.length <= 50`; các số nguyên trong khoảng `[-100, 100]`; độ sâu lớn nhất `<= 50`; **không có danh sách rỗng**.

(Ví dụ gốc trên LeetCode kèm hình minh họa; ở đây ta chỉ ghi lại bằng chữ.)

In [1]:
import json
import random
from collections import deque

examples_raw = [
    ([[1, 1], 2, [1, 1]], 8),
    ([1, [4, [6]]], 17),
]
for raw, expected in examples_raw:
    print(json.dumps(raw), "-> kỳ vọng", expected)

[[1, 1], 2, [1, 1]] -> kỳ vọng 8
[1, [4, [6]]] -> kỳ vọng 17


## 3. Kiến thức nền

### 3.1. Giao diện `NestedInteger`

LeetCode đưa vào **các đối tượng `NestedInteger`**, không phải list Python thuần. Mỗi đối tượng là **một trong hai loại**:

- **Số nguyên đơn:** `isInteger()` trả `True`, `getInteger()` trả số, `getList()` trả `None`.
- **Danh sách:** `isInteger()` trả `False`, `getList()` trả list các `NestedInteger` con, `getInteger()` trả `None`.

Ta tự dựng lớp này (chỉ để chạy thử; LeetCode đã có sẵn, và **không được** tự cài đặt khi nộp).

In [2]:
class NestedInteger:
    def __init__(self, value=None):
        """Nếu không có value: khởi tạo một danh sách rỗng. Ngược lại: một số nguyên đơn bằng value."""
        if value is None:
            self._is_int = False
            self._int = None
            self._list = []
        else:
            self._is_int = True
            self._int = value
            self._list = None

    def isInteger(self):
        return self._is_int

    def add(self, elem):
        """Biến đối tượng thành danh sách (nếu chưa) và thêm elem vào."""
        if self._is_int:
            self._is_int = False
            self._int = None
            self._list = []
        self._list.append(elem)

    def setInteger(self, value):
        self._is_int = True
        self._int = value
        self._list = None

    def getInteger(self):
        return self._int if self._is_int else None

    def getList(self):
        return None if self._is_int else self._list

    def __repr__(self):                    # chỉ để in cho đẹp
        return str(self._int) if self._is_int else "[" + ",".join(map(repr, self._list)) + "]"


a = NestedInteger(5)
b = NestedInteger()
b.add(NestedInteger(1))
b.add(a)
print(a.isInteger(), a.getInteger(), a.getList())      # True 5 None
print(b.isInteger(), b.getInteger(), b.getList())      # False None [1, 5]

True 5 None
False None [1, 5]


### 3.2. Dựng `NestedInteger` từ list Python: `to_nested`

Để viết test dễ, ta cần hàm biến list Python (ví dụ `[1,[4,[6]]]`) thành `NestedInteger`. Quy tắc: số nguyên → `NestedInteger(số)`; danh sách → `NestedInteger()` rồi `add` từng phần tử đã chuyển. Đây cũng là một bài **đệ quy** nhỏ. Đối số cho `depthSumInverse` là **danh sách các `NestedInteger`** của tầng ngoài cùng.

In [3]:
def to_nested(x):
    """int -> NestedInteger(int); list -> NestedInteger chứa các phần tử đã chuyển."""
    if isinstance(x, int):
        return NestedInteger(x)
    ni = NestedInteger()
    for y in x:
        ni.add(to_nested(y))
    return ni


def make_input(raw):
    """raw: list Python tầng ngoài cùng -> list[NestedInteger] đưa cho Solution."""
    return [to_nested(x) for x in raw]


nested = make_input([1, [4, [6]]])
print(nested)                                  # [1, [4,[6]]] (dạng repr)
print([x.isInteger() for x in nested])         # [True, False]
print(nested[1].getList()[1].getList())        # [6]

[1, [4,[6]]]
[True, False]
[6]


### 3.3. Đệ quy trên cấu trúc lồng nhau và `isinstance`

Một danh sách lồng nhau là cấu trúc **tự chứa chính nó**, nên cách tự nhiên để duyệt là **đệ quy**: với mỗi phần tử, nếu là số thì xử lý, nếu là danh sách thì **gọi lại chính hàm** cho từng phần tử con (độ sâu tăng 1). Với list Python thuần ta kiểm tra loại bằng `isinstance(x, int)`; với `NestedInteger` ta dùng `x.isInteger()`.

Thử in từng số cùng độ sâu của nó (độ sâu số ở tầng ngoài cùng là `1`).

In [4]:
def print_depths(x, d):
    """x: list Python hoặc int; d: độ sâu của x nếu x là số."""
    if isinstance(x, int):
        print(f"{x} ở độ sâu {d}")
    else:
        for y in x:
            print_depths(y, d + 1)       # đi vào trong một lớp danh sách


for item in [1, [4, [6]]]:               # mỗi phần tử tầng ngoài cùng có độ sâu 1
    print_depths(item, 1)

1 ở độ sâu 1
4 ở độ sâu 2
6 ở độ sâu 3


### 3.4. `nonlocal` và biến "chung" cho hàm lồng

Khi ta định nghĩa hàm `dfs` **bên trong** `depthSumInverse` và muốn `dfs` **gán lại** một biến của hàm ngoài (ví dụ `maxDepth`), phải khai báo `nonlocal maxDepth`. Nếu không, Python hiểu phép gán là tạo biến **cục bộ mới** và báo `UnboundLocalError` khi đọc trước khi gán. (Chỉ **đọc** hoặc `append` vào list thì không cần.)

In [5]:
def outer():
    count = 0

    def inc_ok():
        nonlocal count
        count += 1                       # có nonlocal: sửa biến của outer

    def inc_bad():
        count += 1                       # không có nonlocal: UnboundLocalError

    inc_ok(); inc_ok()
    try:
        inc_bad()
    except UnboundLocalError as e:
        print("UnboundLocalError:", e)
    return count


print(outer())                           # 2

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value
2


### 3.5. `deque`, `extend` và duyệt theo tầng (BFS)

Để duyệt **từng tầng** ta giữ danh sách phần tử của tầng hiện tại, xử lý hết rồi dựng tầng kế tiếp. `lst.extend(other)` nối **các phần tử** của `other` vào cuối `lst` (khác `append` chỉ thêm đúng một phần tử). Ta sẽ dùng list thường cho từng tầng; hoặc `deque` nếu muốn `popleft`.

In [6]:
a = [1, 2]
a.append([3, 4])
print(a)                                 # [1, 2, [3, 4]]  (append thêm CẢ list như 1 phần tử)
b = [1, 2]
b.extend([3, 4])
print(b)                                 # [1, 2, 3, 4]    (extend thêm TỪNG phần tử)

q = deque([10, 20, 30])
print(q.popleft(), list(q))              # 10 [20, 30]

[1, 2, [3, 4]]
[1, 2, 3, 4]
10 [20, 30]


### 3.6. Tách tổng ra: `Σ a·(M − d + 1) = (M + 1)·Σa − Σ a·d`

Với số `a` ở độ sâu `d`, trọng số là `M − d + 1` (với `M = maxDepth`). Phân phối phép nhân:

`a·(M − d + 1) = a·(M + 1) − a·d`.

Cộng trên mọi số: `Σ a·(M + 1) − Σ a·d = (M + 1)·S − WS`, trong đó `S = Σ a` (tổng thường) và `WS = Σ a·d` (tổng nhân **độ sâu**). Cả hai tính được **trong một lượt duyệt** — chỉ cần `M` ở **cuối**. Kiểm chứng số với `[1,[4,[6]]]`.

In [7]:
pairs = [(1, 1), (4, 2), (6, 3)]                 # (giá trị, độ sâu)
M = max(d for _, d in pairs)
S = sum(a for a, _ in pairs)
WS = sum(a * d for a, d in pairs)
print("M =", M, "S =", S, "WS =", WS)            # M = 3 S = 11 WS = 27
print("trực tiếp:", sum(a * (M - d + 1) for a, d in pairs))        # 17
print("công thức:", (M + 1) * S - WS)                              # 17

M = 3 S = 11 WS = 27
trực tiếp: 17
công thức: 17


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `[[1,1],2,[1,1]]`. Lập bảng từng số, độ sâu và trọng số (`maxDepth = 2`):

| số | độ sâu | trọng số `2 − d + 1` | số × trọng số |
|---|---|---|---|
| 1 | 2 | 1 | 1 |
| 1 | 2 | 1 | 1 |
| 2 | 1 | 2 | 4 |
| 1 | 2 | 1 | 1 |
| 1 | 2 | 1 | 1 |

Tổng `8`. So sánh với bài **339** (trọng số = chính `depth`): `1·2·4 + 2·1 = 10`. Bài này **đảo ngược**: số **nông** nặng hơn.

Khó khăn: để biết trọng số của số `2` ta phải biết `maxDepth = 2`, mà con số này chỉ biết **sau khi nhìn thấy các số sâu hơn** — bên trong các danh sách phía sau. Vì vậy một lượt "tính ngay" là không đủ; cần **hai lượt**, hoặc một **mẹo đại số/đếm lại**.

In [8]:
def depth_weight_pairs(x, d=1):
    """Sinh các cặp (giá trị, độ sâu) từ list Python thuần (dùng để tính tay cho nhanh)."""
    for y in x:
        if isinstance(y, int):
            yield (y, d)
        else:
            yield from depth_weight_pairs(y, d + 1)


raw = [[1, 1], 2, [1, 1]]
pairs = list(depth_weight_pairs(raw))
M = max(d for _, d in pairs)
print(pairs, "maxDepth =", M)
print([a * (M - d + 1) for a, d in pairs], "-> tổng", sum(a * (M - d + 1) for a, d in pairs))

[(1, 2), (1, 2), (2, 1), (1, 2), (1, 2)] maxDepth = 2
[1, 1, 4, 1, 1] -> tổng 8


## 5. Cách 1 — Brute Force / baseline: hai lượt DFS

**Ý tưởng.** Lượt 1: duyệt toàn bộ để tìm `maxDepth`. Lượt 2: duyệt lại, mỗi số `a` ở độ sâu `d` cộng `a · (maxDepth − d + 1)`. Bám sát đề, dễ hiểu, không cần mẹo.

In [9]:
def depth_sum_inverse_two_pass(nestedList):
    def find_max_depth(x, d):
        """Trả về độ sâu lớn nhất của một số nguyên bên trong x (x ở độ sâu d)."""
        if x.isInteger():
            return d
        return max(find_max_depth(y, d + 1) for y in x.getList())

    def weighted_sum(x, d, max_depth):
        if x.isInteger():
            return x.getInteger() * (max_depth - d + 1)
        return sum(weighted_sum(y, d + 1, max_depth) for y in x.getList())

    # các phần tử tầng ngoài cùng có độ sâu 1
    max_depth = max(find_max_depth(x, 1) for x in nestedList)
    return sum(weighted_sum(x, 1, max_depth) for x in nestedList)


for raw, expected in examples_raw:
    print(depth_sum_inverse_two_pass(make_input(raw)), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
17 (kỳ vọng 17 )


**Độ phức tạp (Cách 1).** Gọi `N` là **tổng số phần tử** (số nguyên + số danh sách) và `D` là độ sâu lớn nhất. Hai lượt duyệt, mỗi lượt thăm mỗi phần tử một lần ⇒ **Time `O(N)`**. **Space `O(D)`** cho ngăn xếp đệ quy (`D <= 50`).

**Nút thắt:** phải duyệt **hai lần**. Có thể gộp thành một lượt nếu ta biết biến đổi công thức (Cách 2) hoặc đếm dồn từng tầng (Cách 3).

## 6. Ý tưởng thuật toán / bước đột phá

Có hai "bước ngoặt" khác nhau, cho hai lời giải đẹp:

**(A) Công thức đại số (mục 3.6).** `Σ a·(M − d + 1) = (M + 1)·S − WS`. Trong **một** lượt DFS ta cộng dồn `S`, `WS` và cập nhật `M = max(M, d)`. Cuối cùng áp công thức. Không cần biết `M` trước.

**(B) Cộng dồn theo tầng (BFS).** Số ở độ sâu `d` có trọng số `M − d + 1` = **số tầng từ `d` đến `M`**. Nghĩa là nó được "đếm" đúng **một lần trong mỗi tầng** `d, d+1, ..., M`. Vậy nếu sau mỗi tầng ta **cộng thêm tổng tích lũy của mọi số đã gặp** (các tầng nông hơn + tầng hiện tại) vào kết quả, thì số ở tầng `d` sẽ được cộng đúng ở các tầng `d..M` — chính là trọng số cần. Không cần biết `M` trước vì ta chỉ dừng khi hết tầng.

Ví dụ `[1,[4,[6]]]` (tầng 1: `1`; tầng 2: `4`; tầng 3: `6`): tích lũy sau tầng 1 là `1` ⇒ `ans = 1`; sau tầng 2 là `5` ⇒ `ans = 6`; sau tầng 3 là `11` ⇒ `ans = 17`. ✓

In [10]:
level_sums = [1, 4, 6]                # tổng số ở tầng 1, 2, 3 của [1,[4,[6]]]
running = 0
ans = 0
for k, s in enumerate(level_sums, 1):
    running += s                      # tích lũy mọi số từ tầng 1 đến tầng k
    ans += running                    # mỗi tầng cộng tích lũy hiện có
    print(f"tầng {k}: tổng tầng = {s}, tích lũy = {running}, ans = {ans}")

tầng 1: tổng tầng = 1, tích lũy = 1, ans = 1
tầng 2: tổng tầng = 4, tích lũy = 5, ans = 6
tầng 3: tổng tầng = 6, tích lũy = 11, ans = 17


## 7. Các cách tiếp theo

### Cách 2 — Một lượt DFS với công thức `(maxDepth + 1)·S − WS`

**Giải thích từng đoạn.**

1. `dfs(x, d)`: `x` là `NestedInteger`, `d` là độ sâu của `x`.
2. Nếu `x` là số: cập nhật `maxDepth = max(maxDepth, d)`, rồi `S += a`, `WS += a * d`.
3. Nếu `x` là danh sách: gọi `dfs(y, d + 1)` cho từng phần tử `y` của nó.
4. Sau khi duyệt hết tầng ngoài cùng: kết quả là `(maxDepth + 1) * S - WS`.

(Số ở tầng ngoài cùng có độ sâu `1` — nó nằm trong đúng một danh sách, chính `nestedList`. Bản nộp ở mục 9 cập nhật `maxDepth` ở mọi nút, như lời giải mẫu; mục 10 giải thích vì sao vẫn đúng.)

In [11]:
def depth_sum_inverse_formula(nestedList):
    max_depth = s = ws = 0

    def dfs(x, d):
        nonlocal max_depth, s, ws
        if x.isInteger():
            max_depth = max(max_depth, d)           # chỉ số nguyên mới quyết định maxDepth
            s += x.getInteger()
            ws += x.getInteger() * d
        else:
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, 1)
    return (max_depth + 1) * s - ws


for raw, expected in examples_raw:
    print(depth_sum_inverse_formula(make_input(raw)), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
17 (kỳ vọng 17 )


Ở bản trên ta cập nhật `max_depth` **chỉ khi gặp số nguyên** — chính xác theo đề (`maxDepth` là độ sâu lớn nhất của một **số nguyên**), và vẫn đúng ngay cả khi có danh sách rỗng.

In [12]:
# Code nộp LeetCode cho Cách 2 (NestedInteger do LeetCode cung cấp sẵn)
class SolutionFormula:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        def dfs(x, d):
            nonlocal max_depth, s, ws
            if x.isInteger():
                max_depth = max(max_depth, d)
                s += x.getInteger()
                ws += x.getInteger() * d
            else:
                for y in x.getList():
                    dfs(y, d + 1)

        max_depth = s = ws = 0
        for x in nestedList:
            dfs(x, 1)
        return (max_depth + 1) * s - ws


print(SolutionFormula().depthSumInverse(make_input([1, [4, [6]]])))      # 17

17


**Độ phức tạp Cách 2.** **Time `O(N)`** (một lượt, `N` = tổng số phần tử), **Space `O(D)`** cho ngăn xếp đệ quy (`D` = độ sâu).

### Cách 3 — BFS theo tầng, cộng dồn tích lũy

**Giải thích từng đoạn.**

1. `level` là list các `NestedInteger` của tầng hiện tại (ban đầu chính `nestedList`: các phần tử độ sâu `1`).
2. `running = 0` là **tổng mọi số đã gặp** từ tầng đầu đến tầng hiện tại; `ans = 0`.
3. Mỗi vòng `while level:`: duyệt từng `x` của tầng: nếu là số, `running += x.getInteger()`; nếu là danh sách, `next_level.extend(x.getList())` (các phần tử con sẽ ở tầng sau).
4. Sau khi xong tầng: `ans += running`. Rồi `level = next_level`.
5. Khi hết tầng, `ans` là kết quả (không cần `maxDepth`).

In [13]:
def depth_sum_inverse_bfs(nestedList):
    level = list(nestedList)
    running = 0                              # tổng tích lũy các số từ tầng 1 đến tầng hiện tại
    ans = 0
    while level:
        next_level = []
        for x in level:
            if x.isInteger():
                running += x.getInteger()
            else:
                next_level.extend(x.getList())   # con của x ở tầng kế tiếp
        ans += running                           # mỗi tầng cộng tích lũy hiện có
        level = next_level
    return ans


for raw, expected in examples_raw:
    print(depth_sum_inverse_bfs(make_input(raw)), "(kỳ vọng", expected, ")")

8 (kỳ vọng 8 )
17 (kỳ vọng 17 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionBFS:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        level = list(nestedList)
        running = ans = 0
        while level:
            next_level = []
            for x in level:
                if x.isInteger():
                    running += x.getInteger()
                else:
                    next_level.extend(x.getList())
            ans += running
            level = next_level
        return ans


print(SolutionBFS().depthSumInverse(make_input([[1, 1], 2, [1, 1]])))     # 8

8


**Độ phức tạp Cách 3.** **Time `O(N)`**; **Space `O(W)`** với `W` là bề rộng lớn nhất của một tầng (số phần tử tối đa ở cùng một tầng), tối đa `O(N)`. Điểm hay: **không đệ quy** nên không lo tràn ngăn xếp, và **không cần biết `maxDepth`**.

### Cách 4 — Gom tổng theo độ sâu, rồi nhân trọng số

**Giải thích.** Dùng một `dict` (hoặc list) `by_depth` với `by_depth[d] = tổng các số ở độ sâu d`. DFS một lượt để điền; sau đó `maxDepth = max(by_depth)` và kết quả là `Σ by_depth[d] · (maxDepth − d + 1)`. Ý tưởng khác Cách 2: ta **giữ thông tin theo từng độ sâu** (một "biểu đồ"), rồi áp trọng số ở cuối. Mọi trọng số đều đã biết vì `maxDepth` rõ ràng.

In [15]:
from collections import defaultdict


def depth_sum_inverse_by_depth(nestedList):
    by_depth = defaultdict(int)              # độ sâu -> tổng các số ở độ sâu đó

    def dfs(x, d):
        if x.isInteger():
            by_depth[d] += x.getInteger()
        else:
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, 1)
    max_depth = max(by_depth)
    return sum(total * (max_depth - d + 1) for d, total in by_depth.items())


for raw, expected in examples_raw:
    print(depth_sum_inverse_by_depth(make_input(raw)), "(kỳ vọng", expected, ")")
hist = defaultdict(int)
for a, d in depth_weight_pairs([1, [4, [6]]]):          # biểu đồ tổng theo độ sâu của [1,[4,[6]]]
    hist[d] += a
print(dict(hist))                                       # {1: 1, 2: 4, 3: 6}

8 (kỳ vọng 8 )
17 (kỳ vọng 17 )
{1: 1, 2: 4, 3: 6}


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionByDepth:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        by_depth = {}

        def dfs(x, d):
            if x.isInteger():
                by_depth[d] = by_depth.get(d, 0) + x.getInteger()
            else:
                for y in x.getList():
                    dfs(y, d + 1)

        for x in nestedList:
            dfs(x, 1)
        max_depth = max(by_depth)
        return sum(total * (max_depth - d + 1) for d, total in by_depth.items())


print(SolutionByDepth().depthSumInverse(make_input([1, [4, [6]]])))      # 17

17


**Độ phức tạp Cách 4.** **Time `O(N)`**; **Space `O(D)`** cho `by_depth` (nhiều nhất `D` khóa) cộng `O(D)` ngăn xếp đệ quy.

(Bản `defaultdict` trong cell demo và bản `dict` + `get` trong cell nộp có cùng ý; `max(by_depth)` lấy **khóa lớn nhất**, tức `maxDepth`. Nếu `by_depth` rỗng thì `max` lỗi, nhưng đề đảm bảo có ít nhất một số.)

## 8. TRACE từng bước

Trace **Cách 2** (DFS công thức) trên `[1,[4,[6]]]`: in từng lời gọi `dfs(x, d)` với thụt lề theo độ sâu và các biến `maxDepth`, `S`, `WS`.

In [17]:
def trace_formula(nestedList):
    state = {"max_depth": 0, "s": 0, "ws": 0}

    def dfs(x, d):
        pad = "  " * (d - 1)
        if x.isInteger():
            a = x.getInteger()
            state["max_depth"] = max(state["max_depth"], d)
            state["s"] += a
            state["ws"] += a * d
            print(f"{pad}dfs({a}, d={d}): số -> S={state['s']}, WS={state['ws']}, maxDepth={state['max_depth']}")
        else:
            print(f"{pad}dfs({x}, d={d}): danh sách, đi vào các phần tử ở độ sâu {d + 1}")
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, 1)
    result = (state["max_depth"] + 1) * state["s"] - state["ws"]
    print(f"kết quả = ({state['max_depth']} + 1) * {state['s']} - {state['ws']} = {result}")
    return result


assert trace_formula(make_input([1, [4, [6]]])) == 17

dfs(1, d=1): số -> S=1, WS=1, maxDepth=1
dfs([4,[6]], d=1): danh sách, đi vào các phần tử ở độ sâu 2
  dfs(4, d=2): số -> S=5, WS=9, maxDepth=2
  dfs([6], d=2): danh sách, đi vào các phần tử ở độ sâu 3
    dfs(6, d=3): số -> S=11, WS=27, maxDepth=3
kết quả = (3 + 1) * 11 - 27 = 17


Bảng trace Cách 2 với `[1,[4,[6]]]`:

| Gọi `dfs(x, d)` | loại | `a` | `S` | `WS` | `maxDepth` |
|---|---|---|---|---|---|
| `dfs(1, 1)` | số | 1 | 1 | 1 | 1 |
| `dfs([4,[6]], 1)` | danh sách | | | | |
| `dfs(4, 2)` | số | 4 | 5 | 9 | 2 |
| `dfs([6], 2)` | danh sách | | | | |
| `dfs(6, 3)` | số | 6 | 11 | 27 | 3 |

Kết quả `(3 + 1)·11 − 27 = 17`.

Giờ trace **Cách 3** (BFS) trên `[[1,1],2,[1,1]]` để thấy tích lũy từng tầng.

In [18]:
def trace_bfs(nestedList):
    level = list(nestedList)
    running = ans = 0
    depth = 1
    while level:
        print(f"--- tầng {depth}: {level}")
        next_level = []
        for x in level:
            if x.isInteger():
                running += x.getInteger()
                print(f"    số {x.getInteger()} -> running = {running}")
            else:
                next_level.extend(x.getList())
                print(f"    danh sách {x} -> đẩy {len(x.getList())} con xuống tầng sau")
        ans += running
        print(f"    hết tầng: ans += running ({running}) -> ans = {ans}")
        level = next_level
        depth += 1
    return ans


assert trace_bfs(make_input([[1, 1], 2, [1, 1]])) == 8

--- tầng 1: [[1,1], 2, [1,1]]
    danh sách [1,1] -> đẩy 2 con xuống tầng sau
    số 2 -> running = 2
    danh sách [1,1] -> đẩy 2 con xuống tầng sau
    hết tầng: ans += running (2) -> ans = 2
--- tầng 2: [1, 1, 1, 1]
    số 1 -> running = 3
    số 1 -> running = 4
    số 1 -> running = 5
    số 1 -> running = 6
    hết tầng: ans += running (6) -> ans = 8


Bảng trace Cách 3 với `[[1,1],2,[1,1]]`:

| Tầng | các phần tử | tổng số ở tầng | `running` sau tầng | `ans` sau tầng |
|---|---|---|---|---|
| 1 | `[1,1]`, `2`, `[1,1]` | `2` | `2` | `2` |
| 2 | `1, 1, 1, 1` | `4` | `6` | `8` |

Số `2` ở tầng 1 được cộng vào `ans` ở **cả hai tầng** (`2 + 2 = 4`: trọng số `2` ✓.); mỗi số `1` ở tầng 2 chỉ cộng một lần (trọng số `1` ✓.). Tổng `8`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (một lượt DFS + công thức) — gần với lời giải mẫu và rất ngắn. Lớp `NestedInteger` do LeetCode cung cấp nên **không được** khai báo lại trong bài nộp; ở đây ta chạy thử với lớp tự dựng.

In [19]:
class Solution:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        def dfs(x, d):
            nonlocal max_depth, s, ws
            max_depth = max(max_depth, d)             # d của số sâu nhất sẽ thắng
            if x.isInteger():
                s += x.getInteger()
                ws += x.getInteger() * d
            else:
                for y in x.getList():
                    dfs(y, d + 1)

        max_depth = s = ws = 0
        for x in nestedList:
            dfs(x, 1)
        return (max_depth + 1) * s - ws


sol = Solution()
for raw, expected in examples_raw:
    print(json.dumps(raw), "->", sol.depthSumInverse(make_input(raw)))
    assert sol.depthSumInverse(make_input(raw)) == expected

[[1, 1], 2, [1, 1]] -> 8
[1, [4, [6]]] -> 17


Ở bản nộp này `max_depth` được cập nhật ở **mọi nút** (cả danh sách lẫn số), giống lời giải mẫu. Điều này vẫn đúng vì đề **không có danh sách rỗng** — chi tiết ở mục 10.

## 10. Vì sao thuật toán đúng

- **Độ sâu đúng:** tầng ngoài cùng gọi `dfs(x, 1)`: một số ở tầng này nằm trong đúng một danh sách (chính `nestedList`) nên độ sâu `1` ✓. Mỗi lần đi vào một danh sách ta tăng `d` lên `1` ✓.
- **Công thức đại số:** `Σ a·(M − d + 1) = (M + 1)·Σa − Σ a·d` (phân phối phép nhân). `S` và `WS` là hai tổng tích lũy, cập nhật **một lần cho mỗi số** (mỗi số được thăm đúng một lần vì DFS đi qua mỗi phần tử đúng một lần).
- **`maxDepth` đúng (bản cập nhật ở mọi nút):** một danh sách `x` được gọi với độ sâu `d` thì các phần tử con của nó được gọi với `d + 1`. Vì **không có danh sách rỗng**, `x` có ít nhất một con; đi xuống mãi (cấu trúc hữu hạn) ta tới một số nguyên có độ sâu **lớn hơn** `d`. Vậy giá trị `d` gán cho một danh sách không bao giờ vượt độ sâu của số sâu nhất, nên `max_depth` cuối cùng đúng bằng độ sâu lớn nhất của số nguyên. (Nếu đề cho phép danh sách rỗng, bản này có thể cho `maxDepth` quá lớn; bản chỉ cập nhật khi gặp số vẫn đúng.)
- **BFS cộng dồn đúng:** số ở tầng `d` góp mặt trong `running` ở các tầng `d, d + 1, ..., M` (vì `running` không bao giờ bị trừ), và `ans += running` ở mỗi tầng, nên nó được cộng đúng `M − d + 1` lần = trọng số ✓. Không cần biết `M` trước vì ta chỉ dừng khi hết tầng; số tầng chính là `M`.
- **Hai lượt / gom theo độ sâu:** tính đúng theo định nghĩa sau khi đã biết `maxDepth`.

## 11. Phân tích Time / Space Complexity

Gọi `N` là **tổng số phần tử** (số nguyên + danh sách) trong cấu trúc, `D` là độ sâu lớn nhất, `W` là bề rộng lớn nhất của một tầng.

| Cách | Time | Space |
|---|---|---|
| 1. Hai lượt DFS | `O(N)` | `O(D)` |
| 2. Một lượt DFS + công thức | `O(N)` | `O(D)` |
| 3. BFS cộng dồn theo tầng | `O(N)` | `O(W)` |
| 4. Gom tổng theo độ sâu | `O(N)` | `O(D)` |

Với giới hạn của đề (`<= 50` phần tử tầng ngoài, độ sâu `<= 50`) mọi cách đều tức thì.

## 12. Bộ test

Ta gom các cách vào `impls`. Quan trọng: một **oracle độc lập** — hàm làm việc trên **list Python thuần** (không dùng `NestedInteger`), tính độ sâu bằng cách đếm ngoặc `[`/`]` trên chuỗi JSON của input. Nó khác hẳn các cách trên về dữ liệu và thuật toán.

In [20]:
sol = Solution()
impls = {
    "two_pass": depth_sum_inverse_two_pass,
    "formula": depth_sum_inverse_formula,
    "bfs": depth_sum_inverse_bfs,
    "by_depth": depth_sum_inverse_by_depth,
    "leetcode": sol.depthSumInverse,
}


def oracle_brackets(raw):
    """Oracle độc lập: quét chuỗi JSON, theo dõi độ sâu bằng số dấu [ chưa đóng."""
    text = json.dumps(raw)
    depth = 0
    items = []                                   # (giá trị, độ sâu)
    i = 0
    while i < len(text):
        ch = text[i]
        if ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
        elif ch == "-" or ch.isdigit():
            j = i
            while j + 1 < len(text) and (text[j + 1].isdigit()):
                j += 1
            items.append((int(text[i:j + 1]), depth))
            i = j
        i += 1
    max_d = max(d for _, d in items)
    return sum(v * (max_d - d + 1) for v, d in items)


def check_all(raw, expected, label):
    assert oracle_brackets(raw) == expected, (label, "oracle", oracle_brackets(raw), expected)
    for key, fn in impls.items():
        actual = fn(make_input(raw))
        assert actual == expected, (label, key, actual, expected)
    print("PASS", label)


for idx, (raw, expected) in enumerate(examples_raw, 1):
    check_all(raw, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Một số, toàn bộ cùng tầng (trọng số `1` cho tất cả ⇒ tổng thường), số âm, độ sâu 50, và các danh sách xen kẽ.

In [21]:
check_all([1], 1, "một số duy nhất")
check_all([5, 5, 5], 15, "chỉ một tầng: trọng số 1, tổng thường")
check_all([[3]], 3, "[[3]]: một số, maxDepth = 2, trọng số 1")
check_all([-1, [-2]], -1 * 2 + -2 * 1, "số âm: -1*2 + -2*1 = -4")
check_all([1, [2, [3, [4]]]], 1 * 4 + 2 * 3 + 3 * 2 + 4 * 1, "cầu thang 4 tầng: 4+6+6+4 = 20")
check_all([[1], [1], [1]], 3, "ba danh sách một phần tử")
check_all([0, [0, [0]]], 0, "toàn số 0")
check_all([100, [-100]], 100 * 2 - 100, "100 và -100")

def nest_depth(value, layers):
    """Bọc [value] thêm (layers - 1) lớp nữa: kết quả có đúng `layers` cặp ngoặc."""
    x = [value]
    for _ in range(layers - 1):
        x = [x]
    return x


deep = nest_depth(7, 50)                 # nestedList = deep: số 7 nằm trong 50 danh sách -> độ sâu 50
print("số cặp ngoặc:", json.dumps(deep).count("["))
check_all(deep, 7, "độ sâu 50: số 7 là số duy nhất, trọng số 1")
check_all([1, nest_depth(7, 49)], 1 * 50 + 7, "số 1 ở độ sâu 1 (trọng số 50), số 7 ở độ sâu 50 (trọng số 1)")
check_all([nest_depth(2, 3), 3], 2 * 1 + 3 * 4, "[[[[2]]], 3]: 2 ở độ sâu 4 (trọng số 1), 3 ở độ sâu 1 (trọng số 4)")

PASS một số duy nhất
PASS chỉ một tầng: trọng số 1, tổng thường
PASS [[3]]: một số, maxDepth = 2, trọng số 1
PASS số âm: -1*2 + -2*1 = -4
PASS cầu thang 4 tầng: 4+6+6+4 = 20
PASS ba danh sách một phần tử
PASS toàn số 0
PASS 100 và -100
số cặp ngoặc: 50
PASS độ sâu 50: số 7 là số duy nhất, trọng số 1
PASS số 1 ở độ sâu 1 (trọng số 50), số 7 ở độ sâu 50 (trọng số 1)
PASS [[[[2]]], 3]: 2 ở độ sâu 4 (trọng số 1), 3 ở độ sâu 1 (trọng số 4)


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **dùng `depth` làm trọng số** (đây là bài 339, không phải 364); (b) **dùng `maxDepth` thay vì `maxDepth + 1`** trong công thức (quên `+ 1`).

In [22]:
def wrong_depth_weight(nestedList):
    def dfs(x, d):
        if x.isInteger():
            return x.getInteger() * d              # SAI: trọng số = d (bài 339)
        return sum(dfs(y, d + 1) for y in x.getList())
    return sum(dfs(x, 1) for x in nestedList)


def wrong_formula_no_plus_one(nestedList):
    max_depth = s = ws = 0

    def dfs(x, d):
        nonlocal max_depth, s, ws
        max_depth = max(max_depth, d)
        if x.isInteger():
            s += x.getInteger()
            ws += x.getInteger() * d
        else:
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, 1)
    return max_depth * s - ws                      # SAI: thiếu "+ 1"


raw, expected = [1, [4, [6]]], 17
print(wrong_depth_weight(make_input(raw)), "!=", expected)
print(wrong_formula_no_plus_one(make_input(raw)), "!=", expected)
assert wrong_depth_weight(make_input(raw)) != expected
assert wrong_formula_no_plus_one(make_input(raw)) != expected

27 != 17
6 != 17


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Ta sinh danh sách lồng nhau **đúng ràng buộc**: độ dài tầng ngoài `1..50`, giá trị trong `[-100, 100]`, độ sâu lớn nhất `<= 50`, **không có danh sách rỗng**. Dùng `oracle_brackets` làm chuẩn so với mọi cách. Ba kiểu hình dạng: rộng-nông, hẹp-sâu và trộn.

In [23]:
def gen_list(rng, max_depth, p_list, max_len):
    """Sinh list không rỗng, độ sâu tối đa max_depth (độ sâu của list này tính là 1)."""
    n = rng.randint(1, max_len)
    out = []
    for _ in range(n):
        if max_depth > 1 and rng.random() < p_list:
            out.append(gen_list(rng, max_depth - 1, p_list, max_len))
        else:
            out.append(rng.randint(-100, 100))
    return out


rng = random.Random(364)
for trial in range(2000):
    shape = rng.choice(["wide", "deep", "mixed"])
    if shape == "wide":
        raw = gen_list(rng, rng.randint(1, 4), 0.3, 50)
    elif shape == "deep":
        raw = gen_list(rng, rng.randint(1, 50), 0.8, 2)
    else:
        raw = gen_list(rng, rng.randint(1, 10), 0.5, 5)
    raw = raw[:50]
    assert 1 <= len(raw) <= 50
    ref = oracle_brackets(raw)
    for key, fn in impls.items():
        assert fn(make_input(raw)) == ref, (key, raw)
print("PASS 2000 danh sách lồng nhau ngẫu nhiên đúng ràng buộc")

PASS 2000 danh sách lồng nhau ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `depth` làm trọng số (nhầm với bài 339)

In [24]:
def sai_weight_la_depth(nestedList):
    total = 0
    level = list(nestedList)
    depth = 1
    while level:
        nxt = []
        for x in level:
            if x.isInteger():
                total += x.getInteger() * depth       # SAI: trọng số = depth
            else:
                nxt.extend(x.getList())
        level = nxt
        depth += 1
    return total


print("sai:", sai_weight_la_depth(make_input([[1, 1], 2, [1, 1]])), "| đúng:", Solution().depthSumInverse(make_input([[1, 1], 2, [1, 1]])))

sai: 10 | đúng: 8


**Vì sao sai:** bài này trọng số là `maxDepth − depth + 1`: số **nông** nặng hơn. Dùng `depth` cho `10` (bài 339) thay vì `8`. **Cách sửa:** biết `maxDepth` trước (hai lượt), hoặc dùng công thức `(maxDepth + 1)·S − WS`, hoặc BFS cộng dồn.

### Lỗi 2: Lệch 1 ở trọng số (`maxDepth − depth` thiếu `+ 1`)

In [25]:
def sai_weight_thieu_1(nestedList):
    def depth_max(x, d):
        return d if x.isInteger() else max(depth_max(y, d + 1) for y in x.getList())

    def dfs(x, d, M):
        if x.isInteger():
            return x.getInteger() * (M - d)              # SAI: thiếu "+ 1", số sâu nhất có trọng số 0
        return sum(dfs(y, d + 1, M) for y in x.getList())

    M = max(depth_max(x, 1) for x in nestedList)
    return sum(dfs(x, 1, M) for x in nestedList)


for raw, expected in examples_raw:
    print("sai:", sai_weight_thieu_1(make_input(raw)), "| đúng:", expected)

sai: 2 | đúng: 8
sai: 6 | đúng: 17


**Vì sao sai:** theo đề trọng số là `maxDepth − depth + 1`, nên số **sâu nhất** có trọng số `1` (không phải `0`). Code sai bỏ qua hoàn toàn các số sâu nhất (`[[1,1],2,[1,1]]` cho `2` thay vì `8`). **Cách sửa:** thêm `+ 1`.

Một chi tiết thú vị: nếu bạn đếm độ sâu **từ `0`** (thay vì `1`) **cho cả** `d` **và** `maxDepth` thì công thức `(maxDepth + 1)·S − WS` vẫn cho cùng đáp án (vì lệch `k` ở cả hai vế triệt tiêu: `(M − k + 1)·S − (WS − k·S) = (M + 1)·S − WS`). Ta kiểm chứng ngay. Lỗi chỉ xuất hiện khi bạn **trộn** hai quy ước.

In [26]:
def formula_with_start(nestedList, start):
    max_depth = s = ws = 0

    def dfs(x, d):
        nonlocal max_depth, s, ws
        if x.isInteger():
            max_depth = max(max_depth, d)
            s += x.getInteger()
            ws += x.getInteger() * d
        else:
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, start)                        # start = độ sâu gán cho tầng ngoài cùng
    return (max_depth + 1) * s - ws


for raw, expected in examples_raw:
    a1 = formula_with_start(make_input(raw), 1)
    a0 = formula_with_start(make_input(raw), 0)
    print("start=1:", a1, "| start=0:", a0, "| kỳ vọng:", expected)
    assert a1 == a0 == expected

start=1: 8 | start=0: 8 | kỳ vọng: 8
start=1: 17 | start=0: 17 | kỳ vọng: 17


### Lỗi 3: Quên `nonlocal` (hoặc dùng biến chưa khởi tạo)

In [27]:
def sai_quen_nonlocal(nestedList):
    s = 0

    def dfs(x):
        if x.isInteger():
            s += x.getInteger()               # SAI: gán lại biến của hàm ngoài mà không có nonlocal
        else:
            for y in x.getList():
                dfs(y)

    for x in nestedList:
        dfs(x)
    return s


try:
    sai_quen_nonlocal(make_input([1, [2]]))
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 's' where it is not associated with a value


**Vì sao sai:** trong `dfs`, phép `s += ...` là một **phép gán**, nên Python coi `s` là biến **cục bộ** của `dfs`; đọc nó trước khi gán ⇒ `UnboundLocalError`. **Cách sửa:** thêm `nonlocal s` đầu `dfs`; hoặc để `dfs` **trả về** giá trị thay vì sửa biến ngoài; hoặc dùng một list/dict làm "hộp" (`box[0] += ...`, không cần `nonlocal` vì chỉ sửa nội dung).

### Lỗi 4: Gọi `getInteger()` trên một danh sách / `getList()` trên một số

In [28]:
def sai_khong_kiem_tra_loai(nestedList):
    total = 0
    for x in nestedList:
        total += x.getInteger()               # SAI: x có thể là danh sách -> getInteger() trả None
    return total


try:
    sai_khong_kiem_tra_loai(make_input([1, [2]]))
except TypeError as e:
    print("TypeError:", e)

TypeError: unsupported operand type(s) for +=: 'int' and 'NoneType'


**Vì sao sai:** `getInteger()` trả `None` cho danh sách, và `0 + None` gây `TypeError`. **Cách sửa:** luôn kiểm tra `x.isInteger()` trước; nếu là danh sách mới dùng `x.getList()`. Tương tự, một số nguyên có `getList()` là `None` nên `for y in x.getList()` sẽ lỗi.

### Lỗi 5: Dùng `append` thay `extend` khi dựng tầng kế tiếp (BFS)

In [29]:
def sai_append_thay_extend(nestedList):
    level = list(nestedList)
    running = ans = 0
    while level:
        nxt = []
        for x in level:
            if x.isInteger():
                running += x.getInteger()
            else:
                nxt.append(x.getList())          # SAI: nxt chứa LIST các NestedInteger, không phải NestedInteger
        ans += running
        level = nxt
    return ans


try:
    sai_append_thay_extend(make_input([1, [2]]))
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'list' object has no attribute 'isInteger'


**Vì sao sai:** `append` thêm **cả list con** như một phần tử, nên ở tầng sau `x` là một `list` Python (không có `isInteger`). **Cách sửa:** `nxt.extend(x.getList())` để thêm **từng** `NestedInteger` con.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Cấu trúc lồng nhau + độ sâu**" ⇒ **DFS đệ quy** mang tham số `d`, hoặc **BFS theo tầng**.
- Trọng số phụ thuộc vào đại lượng **toàn cục chưa biết** (`maxDepth`) ⇒ ba lối thoát: duyệt **hai lượt**; **tách công thức** để chỉ cần đại lượng đó ở cuối; hoặc dùng **cộng dồn** để biến "nhân trọng số" thành "cộng lặp".
- `Σ a·(M − d + 1) = (M + 1)·Σa − Σ a·d`: hãy nhớ kỹ thuật **tách tổng thành hai tổng đơn giản**.
- "Cộng dồn tích lũy qua từng tầng" cũng dùng cho các bài như **tổng trọng số nghịch đảo**, hay "mỗi phần tử được đếm theo số tầng còn lại".
- Với `NestedInteger`: luôn gọi `isInteger()` **trước** `getInteger()`/`getList()`.
- Test bằng **oracle độc lập** (đếm ngoặc trên chuỗi JSON) để chắc chắn đã hiểu đúng "độ sâu".

## 15. Biến thể của bài

1. **Nested List Weight Sum (339):** trọng số = `depth`. Một DFS mang `d` là đủ.
2. **Mini Parser (385):** dựng `NestedInteger` từ chuỗi như `"[123,[456,[789]]]"`.
3. **Flatten Nested List Iterator (341):** duyệt lần lượt các số theo thứ tự xuất hiện.
4. **Có danh sách rỗng:** bài này sẽ cần xử lý `maxDepth` cho số nguyên (không tính danh sách rỗng).
5. **Trả về `maxDepth`** hoặc **tổng theo từng độ sâu**.
6. **DFS bằng ngăn xếp tường minh** (không đệ quy) để tránh tràn ngăn xếp khi độ sâu rất lớn.

In [30]:
def depth_sum(nestedList):
    """Bài 339: trọng số = depth."""
    def dfs(x, d):
        if x.isInteger():
            return x.getInteger() * d
        return sum(dfs(y, d + 1) for y in x.getList())
    return sum(dfs(x, 1) for x in nestedList)


def max_depth_of(nestedList):
    def dfs(x, d):
        return d if x.isInteger() else max(dfs(y, d + 1) for y in x.getList())
    return max(dfs(x, 1) for x in nestedList)


def parse_nested(text):
    """Mini Parser (cách đơn giản dùng json): chuỗi -> NestedInteger."""
    return to_nested(json.loads(text))


def depth_sum_inverse_stack(nestedList):
    """Cách 2 nhưng dùng ngăn xếp tường minh (không đệ quy)."""
    stack = [(x, 1) for x in nestedList]
    max_depth = s = ws = 0
    while stack:
        x, d = stack.pop()
        if x.isInteger():
            max_depth = max(max_depth, d)
            s += x.getInteger()
            ws += x.getInteger() * d
        else:
            stack.extend((y, d + 1) for y in x.getList())
    return (max_depth + 1) * s - ws


print(depth_sum(make_input([[1, 1], 2, [1, 1]])))           # 10
print(max_depth_of(make_input([1, [4, [6]]])))              # 3
print(parse_nested("[1,[4,[6]]]"))                          # [1,[4,[6]]]
print(depth_sum_inverse_stack(make_input([1, [4, [6]]])))   # 17
assert depth_sum(make_input([[1, 1], 2, [1, 1]])) == 10
assert depth_sum(make_input([1, [4, [6]]])) == 27
assert depth_sum_inverse_stack(make_input([[1, 1], 2, [1, 1]])) == 8

10
3
[1,[4,[6]]]
17


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai lượt DFS | lượt 1 tìm `maxDepth`, lượt 2 tính tổng | `O(N)` | `O(D)` | dễ hiểu nhất, bám sát đề |
| 2. Một lượt + công thức | `(M + 1)·S − WS` | `O(N)` | `O(D)` | **nên nộp**, ngắn gọn |
| 3. BFS cộng dồn | số ở tầng `d` được cộng ở `M − d + 1` tầng | `O(N)` | `O(W)` | không đệ quy, không cần `maxDepth` |
| 4. Gom theo độ sâu | `by_depth[d]` rồi nhân trọng số | `O(N)` | `O(D)` | cần biểu đồ tổng theo độ sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `NestedInteger(7).getList()`; (b) `to_nested([1, [2]]).getList()[1].isInteger()`; (c) với `M = 4`, `S = 10`, `WS = 25` thì `(M + 1)·S − WS` bằng bao nhiêu? (d) `depth_sum_inverse_bfs(make_input([1, [2]]))`.

**Bài 2 (tính tay).** Tính `depthSumInverse` cho `[[1],[2,[3]],4]`: lập bảng từng số, độ sâu, trọng số.

**Bài 3 (điền TODO).** Hoàn thành `depth_sum_inverse_todo(nestedList)` theo Cách 2.

In [31]:
def depth_sum_inverse_todo(nestedList):
    max_depth = s = ws = 0

    def dfs(x, d):
        # TODO 1: khai báo nonlocal cho max_depth, s, ws
        # TODO 2: nếu x là số: cập nhật max_depth, s += giá trị, ws += giá trị * d
        # TODO 3: nếu x là danh sách: gọi dfs(y, d + 1) cho từng y trong x.getList()
        pass

    # TODO 4: gọi dfs(x, 1) cho từng x trong nestedList
    # TODO 5: trả về (max_depth + 1) * s - ws
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `depth_sum_inverse_bfs2(nestedList)` theo Cách 3 (BFS cộng dồn) **mà dùng `deque`** thay vì list cho tầng hiện tại (gợi ý: xử lý `len(q)` phần tử mỗi vòng).

**Bài 5 (biến thể / mini challenge).** (a) Viết `weight_sum_339(nestedList)` (trọng số = `depth`) bằng BFS và kiểm tra `[[1,1],2,[1,1]] → 10`, `[1,[4,[6]]] → 27`. (b) Viết `level_sums(nestedList)` trả về **list tổng số ở mỗi tầng** (ví dụ `[1,[4,[6]]] → [1, 4, 6]`), rồi dùng nó để tính đáp án bài 364: `Σ level_sums[k] · (D − k)` với `k = 0..D−1`, `D = len(level_sums)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `None`; `False`; `5·10 − 25 = 25`; `[1,[2]]`: tầng 1 có `1` ⇒ `running = 1`, `ans = 1`; tầng 2 có `2` ⇒ `running = 3`, `ans = 4`. Bài 3: nhớ `nonlocal max_depth, s, ws`.
- **Mức 2 (vừa):** Bài 2: số `1` độ sâu `2`, số `2` độ sâu `2`, số `3` độ sâu `3`, số `4` độ sâu `1`; `maxDepth = 3`. Bài 4: `q = deque(nestedList)`; `for _ in range(len(q)): x = q.popleft()`.
- **Mức 3 (sâu):** Bài 5(a): `total += x.getInteger() * depth` với `depth` tăng sau mỗi tầng. 5(b): đếm `level_sums` như BFS nhưng thay vì tích lũy thì `append(tổng tầng)`; trọng số của tầng `k` (đếm từ `0`) là `D − k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert NestedInteger(7).getList() is None
assert to_nested([1, [2]]).getList()[1].isInteger() is False
assert (4 + 1) * 10 - 25 == 25
assert depth_sum_inverse_bfs(make_input([1, [2]])) == 4

# Bài 2: [[1],[2,[3]],4]
# 1: depth 2, w 2 -> 2 ; 2: depth 2, w 2 -> 4 ; 3: depth 3, w 1 -> 3 ; 4: depth 1, w 3 -> 12 ; tổng 21
assert Solution().depthSumInverse(make_input([[1], [2, [3]], 4])) == 2 + 4 + 3 + 12 == 21


# Bài 3
def depth_sum_inverse_todo_ans(nestedList):
    max_depth = s = ws = 0

    def dfs(x, d):
        nonlocal max_depth, s, ws
        if x.isInteger():
            max_depth = max(max_depth, d)
            s += x.getInteger()
            ws += x.getInteger() * d
        else:
            for y in x.getList():
                dfs(y, d + 1)

    for x in nestedList:
        dfs(x, 1)
    return (max_depth + 1) * s - ws


for raw, expected in examples_raw:
    assert depth_sum_inverse_todo_ans(make_input(raw)) == expected


# Bài 4
def depth_sum_inverse_bfs2(nestedList):
    q = deque(nestedList)
    running = ans = 0
    while q:
        for _ in range(len(q)):                 # đúng một tầng
            x = q.popleft()
            if x.isInteger():
                running += x.getInteger()
            else:
                q.extend(x.getList())           # con của x vào cuối hàng đợi: thuộc tầng sau
        ans += running
    return ans


for raw, expected in examples_raw:
    assert depth_sum_inverse_bfs2(make_input(raw)) == expected
assert depth_sum_inverse_bfs2(make_input([[1], [2, [3]], 4])) == 21


# Bài 5
def weight_sum_339(nestedList):
    level = list(nestedList)
    depth = 1
    total = 0
    while level:
        nxt = []
        for x in level:
            if x.isInteger():
                total += x.getInteger() * depth
            else:
                nxt.extend(x.getList())
        level = nxt
        depth += 1
    return total


assert weight_sum_339(make_input([[1, 1], 2, [1, 1]])) == 10
assert weight_sum_339(make_input([1, [4, [6]]])) == 27


def level_sums(nestedList):
    level = list(nestedList)
    sums = []
    while level:
        nxt, total = [], 0
        for x in level:
            if x.isInteger():
                total += x.getInteger()
            else:
                nxt.extend(x.getList())
        sums.append(total)
        level = nxt
    return sums


assert level_sums(make_input([1, [4, [6]]])) == [1, 4, 6]
assert level_sums(make_input([[1, 1], 2, [1, 1]])) == [2, 4]
for raw, expected in examples_raw:
    ls = level_sums(make_input(raw))
    D = len(ls)
    assert sum(ls[k] * (D - k) for k in range(D)) == expected
print("Tất cả đáp án đều PASS")

Tất cả đáp án đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một lượt DFS, công thức `(maxDepth + 1)·S − WS`)

```python
class Solution:
    def depthSumInverse(self, nestedList: list[NestedInteger]) -> int:
        def dfs(x, d):
            nonlocal max_depth, s, ws
            max_depth = max(max_depth, d)
            if x.isInteger():
                s += x.getInteger()
                ws += x.getInteger() * d
            else:
                for y in x.getList():
                    dfs(y, d + 1)

        max_depth = s = ws = 0
        for x in nestedList:
            dfs(x, 1)
        return (max_depth + 1) * s - ws
```

**Checklist**

- [ ] Trọng số là `maxDepth − depth + 1` (nông nặng hơn), không phải `depth`?
- [ ] Số ở tầng ngoài cùng có độ sâu `1` (gọi `dfs(x, 1)`), và `maxDepth` dùng cùng quy ước với `d`?
- [ ] Luôn `isInteger()` trước `getInteger()` / `getList()`? Có `nonlocal` cho biến bị gán lại trong hàm lồng?
- [ ] Nếu dùng BFS: `extend` (không `append`), và `ans += running` **mỗi tầng** (không reset `running`)?
- [ ] Không khai báo lại lớp `NestedInteger` trong bài nộp (LeetCode đã có)?
- [ ] Đã thử `[[1,1],2,[1,1]] → 8`, `[1,[4,[6]]] → 17`, một số duy nhất, số âm, và so với oracle đếm ngoặc?